In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Monitorar ou cancelar um job

Veja a lista das suas cargas de trabalho na [página Workloads](https://quantum.cloud.ibm.com/workloads).

## Ver o status do job
Acesse sua [tabela de Workloads](https://quantum.cloud.ibm.com/workloads) e verifique a coluna Status para saber se um job foi concluído ou falhou.

## Ver o uso restante
Acesse sua [tabela de Instâncias](https://quantum.cloud.ibm.com/instances) e selecione a aba correspondente ao plano para o qual você deseja ver o uso restante. O tempo total utilizado e o tempo total restante no seu plano são exibidos.

## Ver métricas sobre o número de jobs e cargas de trabalho enviados
Acesse a [página de Analytics](https://quantum.cloud.ibm.com/analytics) para ver o número total de jobs enviados, bem como a contagem de cargas de trabalho em batch e em sessão. Observe que você só consegue ver a página de Analytics para contas que você possui ou gerencia.

## Monitorar um job
Use a instância do job para verificar o status do job ou recuperar os resultados chamando o comando apropriado:

|                               |                                                                                                                                                                                                                             |
| ----------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| job.result()                  | Consulta os resultados do job imediatamente após sua conclusão. Os resultados ficam disponíveis após o término do job. Por isso, `job.result()` é uma chamada bloqueante até que o job seja concluído.                      |
| job.job\_id()                 | Retorna o ID que identifica exclusivamente aquele job. Para recuperar os resultados posteriormente é necessário o ID do job. Por isso, recomenda-se salvar os IDs dos jobs que você possa querer recuperar mais tarde.       |
| job.status()                  | Verifica o status do job.                                                                                                                                                                                                   |
| job = service.job(\<job\_id>) | Recupera um job enviado anteriormente. Essa chamada requer o ID do job.                                                                                                                                                     |

<span id="retrieve-later"></span>
## Recuperar os resultados de um job posteriormente
Chame `service.job(\<job\_id>)` para recuperar um job enviado anteriormente. Se você não tiver o ID do job, ou quiser recuperar vários jobs de uma vez — incluindo jobs de QPUs (unidades de processamento quântico) desativadas —, chame `service.jobs()` com filtros opcionais. Consulte [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` também retorna jobs executados pelo pacote `qiskit-ibm-provider`, que está descontinuado. Jobs enviados pelo pacote mais antigo (também descontinuado) `qiskit-ibmq-provider` não estão mais disponíveis.

### Exemplo
Este exemplo retorna os 10 jobs de runtime mais recentes executados em `my_backend`:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>